# មេរៀន 07 - លំនាំរចនាផែនការ

កំណត់សម្គាល់នេះបង្ហាញពី **លំនាំរចនាផែនការ** សម្រាប់ភ្នាក់ងារ AI ដែលប្រើ Microsoft Agent Framework។
អ្នកនឹងរៀនពីរបៀបបំបែកសំណើធ្វើដំណើរដែលស្មុគស្មាញទៅជាកិច្ចការតូចៗដែលមានរចនាសម្ព័ន្ធ, មอบหมายពួកវាទៅឲ្យភ្នាក់ងារជាអ្នកជំនាញ,
និងអនុវត្តផែនការដែលទទួលបាន — ទាំងអស់នេះប្រើលទ្ធផលដែលមានរចនាសម្ព័ន្ធ ដែលគាំទ្រដោយម៉ូឌែល Pydantic។


## ការតំឡើង


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ការបំបែកភារកិច្ច

ការបំបែកភារកិច្ចគឺជាគ្រឹះនៃលំនាំការរចនាការធ្វើផែនការ។ ជំនួសនៃការស្នើឱ្យភ្នាក់ងារតែមួយដើម្បី
ដោះស្រាយសំណើដែលស្មុគស្មាញពីចាប់ផ្តើមដល់ចប់ យើងបំបែកបញ្ហាទៅជាឯកតាតូចៗដែលបានកំណត់យ៉ាងច្បាស់ **ភារកិច្ចរង**.
ភារកិច្ចរងនីមួយៗត្រូវបានផ្ដល់ឱ្យភ្នាក់ងារដែលជាពិសេស (ឧទាហរណ៏៖ ជើងយន្តហោះ, សណ្ឋាគារ, សកម្មភាព) ជាមួយ
អាទិភាពច្បាស់លាស់ និងលំដាប់ការពឹងផ្អែក។

This approach provides several benefits:
- **ភាពច្បាស់**: ភារកិច្ចរងនីមួយៗមានការទទួលខុសត្រូវតែមួយ។
- **ការដំណើរការដោយសមមូល**: ភារកិច្ចរងដែលឯករាជ្យអាចដំណើរការនៅពេលតែមួយ។
- **ភាពទុកចិត្តបាន**: ករណីបរាជ័យត្រូវបានដាច់ឡែកចេញនៅលើភារកិច្ចរងនីមួយៗ។
- **ការតាមដានថវិកា**: ការចំណាយត្រូវបានប៉ាន់ប្រមាណសម្រាប់ភារកិច្ចរងនីមួយៗ ហើយរួមបូកចូលគ្នា។


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ការបង្កើតភ្នាក់ងាររៀបចំផែនការជាមួយលទ្ធផលដែលមានរចនាសម្ព័ន្ធ

ភ្នាក់ងាររៀបចំផែនការនេះមានតួនាទីជា **អ្នកសម្របសម្រួលតុទទួលភ្ញៀវ**។ ដោយទទួលបានសំណើធ្វើដំណើរកម្រិតខ្ពស់ វា
ផលិត `TravelPlan` ដែលមានរចនាសម្ព័ន្ធ — បំបែកសំណើទៅជាកិច្ចការតូចៗ កំណត់អាទិភាព,
និងសម្គាល់ការពឹងផ្អែកគ្នា ដើម្បីឱ្យបុគ្គលិកផ្តល់សេវា ឬស្រទាប់អនុវត្តអាចអនុវត្តការងារនោះបាន។


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ការអនុវត្តផែនការជាមួយឧបករណ៍ជំនាញ

បន្ទាប់ពីភ្នាក់ងារមុខទទួលបានបង្កើតផែនការដែលមានរចនាសម្ព័ន្ធ, **ភ្នាក់ងារបំរើ** នឹងអនុវត្តវា.
ឧបករណ៍ជំនាញនីមួយៗ ដំណើរការកិច្ចការរងមួយប្រភេទ (យន្តហោះ, សណ្ឋាគារ, សកម្មភាព). ភ្នាក់ងារបំរើ នឹងវិលតាមកិច្ចការរងក្នុងផែនការតាមលំដាប់អាស្រ័យភាព ហើយផ្ញើកិច្ចការនីមួយៗទៅឧបករណ៍ដែលសមស្រប។


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## សេចក្តីសង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនអំពី **គំរូរចនាផែនការ** សម្រាប់ភ្នាក់ងារ AI:

1. **ការបំបែកភារកិច្ច** — ភ្នាក់ងាររៀបចំផែនការមុខតុ បំបែកសំណើធ្វើដំណើរដែលស្មុគស្មាញចេញទៅ
   ជាកិច្ចការតូចៗដែលមានរចនាសម្ព័ន្ធដោយប្រើម៉ូដែល Pydantic ហើយចាត់តាំងមួយចំពោះភ្នាក់ងារជំនាញដោយមានអាទិភាព
   និងការពឹងផ្អែក។
2. **លទ្ធផលដែលមានរចនាសម្ព័ន្ធ** — ដោយបញ្ជូន `response_format` ភ្នាក់ងារត្រឡប់នូវដែលបានផ្ទៀងផ្ទាត់
   `TravelPlan` វត្ថុជំនួសអត្ថបទទ្រង់ទ្រាយសេរី ធ្វើឲ្យដំណើរការផ្នែកក្រោយទុកចិត្តបាន។
3. **ការអនុវត្តផែនការ** — ភ្នាក់ងារបម្រើ ធ្វើការឆ្លងកាត់កិច្ចការតូចៗទាំងនោះ ដោយប្រើឧបករណ៍ជំនាញ
   (`book_flight`, `reserve_hotel`, `book_activity`) ដើម្បីអនុវត្តផែនការ និងរាយការណ៍លទ្ធផល។

គំរូនេះបំបែក *អ្វីដែលត្រូវធ្វើ* (ការរៀបចំផែនការ) ពី *របៀបធ្វើវា* (ការអនុវត្ត) ធ្វើឲ្យភ្នាក់ងារមានរចនាសម្ព័ន្ធបំបែកជាផ្នែក (modular), អាចសាកល្បងបាន និងងាយស្រួលក្នុងការពង្រីក។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារ​នេះ​ត្រូវបាន​បកប្រែ​ដោយប្រើ​សេវាកម្ម​បកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលយើងខិតខំសម្រាប់ភាពត្រឹមត្រូវ សូមយល់ថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬមិនត្រឹមត្រូវ។ សូមចាត់ទុកឯកសារដើមក្នុងភាសាដើមជាប្រភពដែលត្រូវទុកជាការយោង។ សម្រាប់ព័ត៌មានសំខាន់ៗ មានការផ្ដល់អនុសាសន៍ឱ្យប្រើការបកប្រែដោយអ្នកបកប្រែមនុស្សវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកអត្ថន័យខុសដែលបណ្តាលមកពីការប្រើប្រាស់ការបកប្រែនេះ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
